# Robot Arm Pick and Place - Complete Visualization

## 🎯 What This Notebook Does

This notebook creates a **complete robot arm simulation** that shows:
- ✅ A real robot arm (Panda robotic manipulator)
- ✅ A gripper that opens and closes
- ✅ The arm moving to pick up the red cube
- ✅ The arm placing it next to the blue cube
- ✅ Everything recorded in a video!

## 🔍 Why the Original Didn't Show the Robot

The original notebook had a problem:
- It planned robot actions (pick, place)
- But it only moved the cubes directly
- No actual robot arm was loaded or visualized
- The cubes just "floated" to their destinations!

## ✨ What We'll Fix

This notebook will:
1. Load a proper robot arm model
2. Show the arm moving to the cube
3. Show the gripper closing to grab it
4. Show the arm lifting and moving
5. Show the gripper opening to release

Let's begin! 🚀

---

## Part 1: Install Dependencies

In [ ]:
# Install all required packages
!pip install -q pybullet imageio imageio-ffmpeg numpy pillow

print("✅ All dependencies installed!")
print("\n📦 Installed:")
print("   - PyBullet: Physics simulation with robot models")
print("   - imageio: Video creation")
print("   - numpy: Math operations")

: 

---

## Part 2: Import Libraries

In [ ]:
import pybullet as p
import pybullet_data
import numpy as np
import imageio
from PIL import Image
from IPython.display import display, Video
import time
import math

print("✅ All imports successful!")

---

## Part 3: Create Robot Arm Simulation

Now we'll create a simulation with a **real robot arm**!

In [ ]:
class RobotArmSimulation:
    """
    A complete robot arm simulation with pick and place capabilities.
    
    This class:
    - Loads a Panda robot arm
    - Controls the gripper
    - Moves the arm to pick and place objects
    - Records video of all actions
    """
    
    def __init__(self, use_gui=False):
        """
        Initialize the robot arm simulation.
        
        Args:
            use_gui: If True, shows a window with the simulation
        """
        print("🤖 Initializing Robot Arm Simulation...\n")
        
        # Connect to PyBullet
        if use_gui:
            self.client = p.connect(p.GUI)
        else:
            self.client = p.connect(p.DIRECT)
        
        print("   ✅ Connected to PyBullet")
        
        # Set up physics
        p.setAdditionalSearchPath(pybullet_data.getDataPath())
        p.setGravity(0, 0, -9.8)
        p.setTimeStep(1./240.)  # Smooth physics
        
        print("   ✅ Physics configured")
        
        # Storage for video frames
        self.frames = []
        
        # Load the scene
        self._load_scene()
        
        print("\n✅ Robot arm simulation ready!\n")
    
    def _load_scene(self):
        """
        Load the robot arm, table, and cubes.
        """
        print("🎬 Loading scene...")
        
        # Load table/ground
        self.plane_id = p.loadURDF("plane.urdf")
        print("   ✅ Loaded table/ground")
        
        # Load robot arm (Panda)
        # The Panda arm is a 7-DOF robot with a gripper
        self.robot_id = p.loadURDF(
            "franka_panda/panda.urdf",
            basePosition=[0, 0, 0],
            useFixedBase=True
        )
        print("   ✅ Loaded Panda robot arm")
        
        # Get gripper finger indices
        # The Panda has two finger joints for the gripper
        #The choice of [9, 10] is specific to the Panda robot's URDF structure - they're the exact indices where the gripper fingers are defined in that particular robot model
        self.gripper_finger_joints = [9, 10]
        
        # Set initial robot configuration
        # This puts the arm in a neutral position
        #This list contains 7 numbers (in radians) that define the angle for each of the 7 arm joints:
        #This specific pose is chosen because it:
        #Keeps the arm upright and ready - Not collapsed or fully extended
        #Positions the gripper at a reachable height - Around 0.6-0.7m above the table
        #Avoids self-collisions - The arm isn't folded into itself
        #Provides good workspace access - Can easily reach the cubes positioned at (0.4, ±0.2, 0.65)
        #This is also a common starting pose for robot arms in simulation and real-world applications. It is published by the manufacturer and is a good starting point for most tasks.

        self.rest_joints = [0, -0.5, 0, -2.0, 0, 1.5, 0.785]
        for i, joint_pos in enumerate(self.rest_joints):
            p.resetJointState(self.robot_id, i, joint_pos)
        
        print("   ✅ Robot positioned")
        
        # Load RED cube (on the left)
        #The cube itself has size! If we put it at Z = 0, it would be half buried in the ground:
        # Panda robot specifications:
        #min_reach_height = 0.10 meters (10cm)
        #max_reach_height = 1.20 meters (120cm)
        #comfortable_height = 0.60-0.70 meters (60-70cm)

# At 63cm: Perfect working height! ✅
        self.red_cube = p.loadURDF(
            "cube_small.urdf",
            basePosition=[0.4, -0.2, 0.65],  # Position for robot to reach
            globalScaling=0.8 #The Panda gripper can open to about 4-5cm. A 4cm cube fits perfectly:

        )
        p.changeVisualShape(self.red_cube, -1, rgbaColor=[1, 0, 0, 1])
        print("   ✅ Loaded RED cube at (0.4, -0.2, 0.65)")
        
        # Load BLUE cube (on the right)
        self.blue_cube = p.loadURDF(
            "cube_small.urdf",
            basePosition=[0.4, 0.2, 0.65],
            globalScaling=0.8
        )
        p.changeVisualShape(self.blue_cube, -1, rgbaColor=[0, 0, 1, 1])
        print("   ✅ Loaded BLUE cube at (0.4, 0.2, 0.65)")
        
        # Let physics settle
        for _ in range(100):
            p.stepSimulation()
        
        print("   ✅ Physics settled")
    
    def move_arm_to_position(self, target_pos, num_steps=60):
        """
        Move the robot arm's end effector to a target position.
        
        This uses inverse kinematics to calculate joint angles.
        At 240 Hz physics: 60 steps ÷ 240 = 0.25 seconds (quarter second)
        # Fast enough to look responsive
        #Slow enough to look natural
        #A human takes about 0.3-0.5 seconds to reach for something - we're similar!

        
        Args:
            target_pos: (x, y, z) position to move to
            num_steps: How many simulation steps to use (smoothness)
        """
        # Get end effector link index (the part of the arm that moves)
        #The end effector is the part of the robot that actually interacts with the object
        #For the Panda robot, it's the last link in the arm chain
        #This is a fixed index for the Panda robot model    after the index 9, 10 the fingers
        end_effector_index = 11
        
        # Use inverse kinematics to find joint angles
        #joint_poses is a list of angles (in radians) for ALL joints!
        #This is because the inverse kinematics solver needs to know the angles for all joints to move the arm correctly
        #It's like a recipe that tells the robot how to move its arm from the current position to the target position
        #The solver will adjust the angles of the joints to reach the target position
        #It will try to find a solution that works, even if it's not the shortest path
        #This is why we need to provide all 7 joint angles
        #Actually returns MORE than 7 values (includes finger joints too), but we only use the first 7 for the arm.

        joint_poses = p.calculateInverseKinematics(
            self.robot_id,
            end_effector_index,
            target_pos,
            maxNumIterations=100,
            residualThreshold=0.001 #  "Gripper gets within 1mm of target",Precise enough to grab objects ✅


        )
        

        # Move smoothly to the target
        for step in range(num_steps):
            # Set joint positions
            for i in range(7):  # 7 arm joints What it does: Send commands to joints 0, 1, 2, 3, 4, 5, 6 (the 7 arm joints)
                p.setJointMotorControl2(
                    self.robot_id,
                    i, #which joint to control
                    p.POSITION_CONTROL, #"Move to this position and stay there"
                    targetPosition=joint_poses[i],
                    force=500 #500 N → Strong enough to move the arm ✅     
                )
            
            # Step physics
            p.stepSimulation()
            
            # Record frame every few steps for smooth video
            #This is to make the video smooth and not too fast
            #We only record every 3 steps (1/8th of a second)
            #This is a balance between video quality and file size
            #If we recorded every step, the video would be too big
            #If we recorded every 10 steps, the video would be too slow
            #So we record every 3 steps to get a good balance   
            if step % 3 == 0:
                self.capture_frame()
    
    def control_gripper(self, close=True, num_steps=30):
        """
        Open or close the gripper.
        Physics runs at 240 Hz (240 steps per second)
        30 steps ÷ 240 Hz = 0.125 seconds (1/8th of a second)
        #That's fast enough to look responsive but slow enough to look smooth
        #Real-world analogy:
        #How fast does a human close their hand to grab something? About 0.1 to 0.2 seconds - that's exactly what 30 steps gives us! 
        Args:
            close: If True, close gripper; if False, open it
            num_steps: How many steps to animate
        """
        
        #Let's think about cube size:
        #Our cube: globalScaling=0.8 × standard cube (5cm) = 4 cm wide
        #If gripper opens to 4 cm, it can fit around the cube perfectly!
        
        target_position = 0.0 if close else 0.04  # Gripper width
        
        for step in range(num_steps):
            for finger_joint in self.gripper_finger_joints:
                #Sends the same command to both gripper fingers , left finger = joint 9 and right finger = joint 10
                #This is because the gripper is a simple mechanism with two fingers that move in sync
                p.setJointMotorControl2(
                    self.robot_id,
                    finger_joint,
                    p.POSITION_CONTROL,#"Move to this position and stay there"

                    targetPosition=target_position,
                    force=50 #50 N  → Like a normal firm grip ✅ (our choice)

                )
            
            p.stepSimulation()
            
            if step % 2 == 0:
                self.capture_frame()
    
    def create_constraint(self, obj_id):
        """
        Create a constraint to 'attach' object to gripper.
        This simulates grasping.
        A constraint is like an invisible glue or virtual rope that connects two objects together.

        """
        #This line gets where the gripper currently is in 3D space.
        #p.getLinkState()
        #What it does: Returns information about a specific part of the robot
        #Parameters:
        #self.robot_id - Which robot?
        #11 - Which part? (link 11 = end effector/gripper center)
        #Returns: A tuple with LOTS of information:
        #(    position,           # [0] - (x, y, z) position    orientation,        # [1] - (x, y, z, w) quaternion    local_inertia,      # [2] - physics stuff    local_frame_pos,    # [3] - more physics    local_frame_orn,    # [4] - more physics    world_frame_pos,    # [5] - even more...    world_frame_orn     # [6] - even more...)
        #[:2] - Taking Only First Two
        #We only need the first 2 items (position and orientation), so we slice:
        #We need to know where the gripper is so we can create a constraint at the correct location!

        gripper_pos, gripper_orn = p.getLinkState(self.robot_id, 11)[:2]
        #This line gets where the object currently is in 3D space.
        #p.getBasePositionAndOrientation()
        #What it does: Returns the position and orientation of an object
        #Parameters:
        #obj_id - Which object?
        #Returns: A tuple with 2 items: 
        obj_pos, obj_orn = p.getBasePositionAndOrientation(obj_id)
        
        self.constraint = p.createConstraint(
            self.robot_id,#Which robot?             
            11,  # end effector link
            obj_id,#Which object? the child object
            -1,#Which part of the object? the whole object  
            p.JOINT_FIXED,#What type of constraint? object is fixed to the robot
            [0, 0, 0],#Where on the robot?
            [0, 0, -0.02],#Where on the object? 2cm below the gripper
            [0, 0, 0]
        )
    
    def remove_constraint(self):
        """Remove the grasping constraint to release object.
        This function breaks the connection between the gripper and the object so the object can fall or stay where we place it.
        
        """
        if hasattr(self, 'constraint'):
            p.removeConstraint(self.constraint)
            delattr(self, 'constraint')
    
    def pick_and_place(self, obj_id, pick_pos, place_pos):
        """
        Complete pick and place sequence.
        
        Args:
            obj_id: PyBullet ID of object to pick
            pick_pos: (x, y, z) position to pick from
            place_pos: (x, y, z) position to place at
        """
        print("\n🤖 Executing Pick and Place...\n")
        
        # Step 1: Open gripper
        print("   1. Opening gripper...")
        self.control_gripper(close=False)
        
        # Step 2: Move above pick position
        print("   2. Moving above object...")
        above_pick = (pick_pos[0], pick_pos[1], pick_pos[2] + 0.15)
        self.move_arm_to_position(above_pick)
        
        # Step 3: Move down to pick position
        print("   3. Moving down to grasp...")
        self.move_arm_to_position(pick_pos)
        
        # Step 4: Close gripper
        print("   4. Closing gripper...")
        self.control_gripper(close=True)
        
        # Step 5: Attach object
        print("   5. Grasping object...")
        self.create_constraint(obj_id)
        
        # Step 6: Lift object
        print("   6. Lifting object...")
        self.move_arm_to_position(above_pick)
        
        # Step 7: Move above place position
        print("   7. Moving to place location...")
        above_place = (place_pos[0], place_pos[1], place_pos[2] + 0.15)
        self.move_arm_to_position(above_place)
        
        # Step 8: Lower to place position
        print("   8. Lowering object...")
        self.move_arm_to_position(place_pos)
        
        # Step 9: Release object
        print("   9. Releasing object...")
        self.remove_constraint()
        
        # Step 10: Open gripper
        print("   10. Opening gripper...")
        self.control_gripper(close=False)
        
        # Step 11: Retract
        print("   11. Retracting arm...")
        self.move_arm_to_position(above_place)
        
        print("\n✅ Pick and place complete!\n")
    
    def capture_frame(self):
        """
        Capture one frame from the virtual camera.
        """
        # Set up camera view
        view_matrix = p.computeViewMatrixFromYawPitchRoll(
            cameraTargetPosition=[0.3, 0, 0.7],
            distance=1.2,
            yaw=45,
            pitch=-20,
            roll=0,
            upAxisIndex=2
        )
        
        proj_matrix = p.computeProjectionMatrixFOV(
            fov=60,
            aspect=640/480,
            nearVal=0.1,
            farVal=10.0
        )
        
        # Capture image
        img_arr = p.getCameraImage(
            640, 480,
            viewMatrix=view_matrix,
            projectionMatrix=proj_matrix,
            shadow=True,
            renderer=p.ER_BULLET_HARDWARE_OPENGL
        )
        
        # Extract RGB
        w, h, rgba, _, _ = img_arr
        rgba = np.array(rgba, dtype=np.uint8).reshape(h, w, 4)
        rgb = rgba[:, :, :3]
        
        self.frames.append(rgb)
    
    def save_video(self, filename="robot_arm_demo.mp4", fps=30):
        """
        Save all captured frames as a video.
        
        Args:
            filename: Output video filename
            fps: Frames per second
        """
        print(f"\n💾 Saving video with {len(self.frames)} frames...")
        imageio.mimsave(filename, self.frames, fps=fps)
        print(f"✅ Video saved: {filename}")
        print(f"   Duration: {len(self.frames)/fps:.2f} seconds\n")
        return filename
    
    def close(self):
        """Clean up simulation."""
        p.disconnect()
        print("✅ Simulation closed")

print("✅ RobotArmSimulation class defined!")

---

## Part 4: Run the Complete Simulation

Now let's create the simulation and execute the pick-and-place task!

In [ ]:
# Create the simulation
sim = RobotArmSimulation(use_gui=False)

print("📸 Capturing initial scene...")
for _ in range(30):  # 1 second of initial state
    sim.capture_frame()
print("✅ Initial state recorded\n")

### Display Initial Scene

In [ ]:
# Show what the scene looks like
print("🖼️ Initial Scene:\n")
display(Image.fromarray(sim.frames[0]))

---

## Part 5: Execute Pick and Place

Now the exciting part - watch the robot arm pick up the red cube and place it next to the blue one!

In [ ]:
# Get current cube positions
red_pos, _ = p.getBasePositionAndOrientation(sim.red_cube)
blue_pos, _ = p.getBasePositionAndOrientation(sim.blue_cube)

print(f"📍 Red cube position: ({red_pos[0]:.2f}, {red_pos[1]:.2f}, {red_pos[2]:.2f})")
print(f"📍 Blue cube position: ({blue_pos[0]:.2f}, {blue_pos[1]:.2f}, {blue_pos[2]:.2f})\n")

# Calculate place position (next to blue cube)
place_pos = (blue_pos[0], blue_pos[1] - 0.15, blue_pos[2])  # 15cm to the left of blue

print(f"🎯 Target position: ({place_pos[0]:.2f}, {place_pos[1]:.2f}, {place_pos[2]:.2f})\n")

# Execute the pick and place!
sim.pick_and_place(
    obj_id=sim.red_cube,
    pick_pos=red_pos,
    place_pos=place_pos
)

### Record Final State

In [ ]:
print("📸 Capturing final scene...")
for _ in range(30):  # 1 second of final state
    sim.capture_frame()
print(f"✅ Final state recorded")
print(f"\n📊 Total frames captured: {len(sim.frames)}")

---

## Part 6: Save and Display Video

Let's create the video and watch the robot in action!